<a href="https://colab.research.google.com/github/rishav2404/qc-ai-hub/blob/main/qc_ai_hub_workbench.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### In this flow, we are optimizing the open source MobileNet model for the Samsung Galaxy S25 (Family), which uses a TFLite runtime
Get API key from : https://workbench.aihub.qualcomm.com/account

In [ ]:
## Account setup
!pip3 install qai-hub

import os
from google.colab import userdata
my_secret = userdata.get('QC_API_KEY')
os.environ['QC_API_KEY'] = my_secret

!qai-hub configure --api_token $QC_API_KEY

2026-10-16 14:24:38.766 - INFO - Enabling verbose logging.
/usr/local/lib/python3.13/dist-packages/qai_hub/_cli.py:484: UserWarning: Overwriting configuration: /root/.qai_hub/client.ini (previous configuration saved to /root/.qai_hub/client.ini.bak)
  warnings.warn(
qai-hub configuration saved to /root/.qai_hub/client.ini
==================== /root/.qai_hub/client.ini ====================
[api]
api_token = ilsx7q78wcwso22v51lwntxz633updf56zlle3ba
api_url = https://workbench.aihub.qualcomm.com
web_url = https://workbench.aihub.qualcomm.com
verbose = True
client_mode = cli




#### **Setip Dependencies**

In [ ]:
!pip3 install 'qai-hub[torch]'

#### **Import and Export model**

In [ ]:
# Import torch and pre-trained MobileNet
import torch
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights
torch_model = mobilenet_v2(weights=MobileNet_V2_Weights.IMAGENET1K_V1)
torch_model.eval()

# Export model
input_shape = (1, 3, 224, 224)
example_input = torch.rand(input_shape)
with torch.no_grad():
    exported_torch_model = torch.export.export(torch_model, (example_input,))

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 86.3MB/s]


#### **Select device and compile**

In [ ]:
import qai_hub as hub
device = hub.Device("Samsung Galaxy S25 (Family)")

# Compile model
compile_job = hub.submit_compile_job(
        model=exported_torch_model,
        device=device,
        input_specs=dict(image=input_shape),
        options="--target_runtime tflite"
)
target_model = compile_job.get_target_model()


Uploading tmp7b66dm03.pt2: 100%|██████████| 14.7M/14.7M [00:00<00:00, 38.4MB/s]


Scheduled compile job (jg9oo188g) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jg9oo188g/



#### **Submit an Inference jobb**

In [ ]:
import requests
import numpy as np
from PIL import Image

sample_image_url = (
    "https://qaihub-public-assets.s3.us-west-2.amazonaws.com/apidoc/input_image1.jpg"
)

# Convert image to numpy array to match input specs
response = requests.get(sample_image_url, stream=True)
response.raw.decode_content = True
image = Image.open(response.raw).resize((224, 224))
input_array = np.expand_dims(
    np.transpose(np.array(image, dtype=np.float32) / 255.0, (2, 0, 1)), axis=0
)

# Run inference using input image
inference_job = hub.submit_inference_job(
    model=target_model,
    device=device,
    inputs=dict(image=[input_array]),
)
on_device_output = inference_job.download_output_data()

Uploading dataset: 100%|██████████| 150k/150k [00:00<00:00, 782kB/s]


Scheduled inference job (jp1ool375) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jp1ool375/

Waiting for inference job (jp1ool375) completion. Type Ctrl+C to stop waiting at any time.
    ✅ SUCCESS                          


tmpwh6yl5hw.h5: 100%|██████████| 14.5k/14.5k [00:00<00:00, 147kB/s]


#### **Validate model accuracy**

In [ ]:
# Post-process the on-device output
output_name = list(on_device_output.keys())[0]
out = on_device_output[output_name][0]
on_device_probabilities = np.exp(out) / np.sum(np.exp(out), axis=1)

# Read the class labels for imagenet
sample_classes = "https://qaihub-public-assets.s3.us-west-2.amazonaws.com/apidoc/imagenet_classes.txt"
response = requests.get(sample_classes, stream=True)
response.raw.decode_content = True
categories = [s.decode().strip() for s in response.raw]

# Display input image
from IPython import display
display.Image(sample_image_url)

# Print top 5 predictions for the on-device model
print("Top-5 On-Device predictions:")
top5_classes = np.argsort(on_device_probabilities[0], axis=0)[-5:]
for c in reversed(top5_classes):
    print(f"{c} {categories[c]:20s} {on_device_probabilities[0][c]:>6.1%}")

Top-5 On-Device predictions:
968 cup                   71.3%
504 coffee mug            16.4%
967 espresso               7.8%
809 soup bowl              1.4%
659 mixing bowl            1.3%


#### **Profile on-device**

In [ ]:
# Submit profile job
# profile_job = hub.submit_profile_job(
#     model=target_model,
#     device=device,
# )

profile_job = hub.submit_inference_job(
    model=target_model,
    device=device,
    profile=True
)

Input data was not specified. Random data will be used for inputs.
Scheduled inference job (jglwwzo2p) successfully. To see the status and results:
    https://workbench.aihub.qualcomm.com/jobs/jglwwzo2p/

